In [8]:
import os
print("--- KAGGLE FOLDER X-RAY ---")
print("Main Folders:", os.listdir("/kaggle/input"))

total_files = 0
for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        total_files += 1
        if total_files <= 5:
            print(f"Found file: {file} in path: {root}")
            
print(f"\nTotal files found in the entire drive: {total_files}")

--- KAGGLE FOLDER X-RAY ---
Main Folders: ['datasets']
Found file: v_HorseRace_g23_c05.avi in path: /kaggle/input/datasets/pevogam/ucf101/UCF101/UCF-101/HorseRace
Found file: v_HorseRace_g18_c05.avi in path: /kaggle/input/datasets/pevogam/ucf101/UCF101/UCF-101/HorseRace
Found file: v_HorseRace_g10_c01.avi in path: /kaggle/input/datasets/pevogam/ucf101/UCF101/UCF-101/HorseRace
Found file: v_HorseRace_g18_c04.avi in path: /kaggle/input/datasets/pevogam/ucf101/UCF101/UCF-101/HorseRace
Found file: v_HorseRace_g16_c01.avi in path: /kaggle/input/datasets/pevogam/ucf101/UCF101/UCF-101/HorseRace

Total files found in the entire drive: 13327


In [9]:
!pip install torchaudio opencv-python

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class AdaptiveGatingMechanism(nn.Module):
    def __init__(self, visual_dim: int, audio_dim: int, joint_dim: int):
        """
        Adaptive Gating Mechanism (CMRF-Net inspired)
        Dynamically weights audio and visual tokens based on context and noise.
        """
        super(AdaptiveGatingMechanism, self).__init__()
        
        self.visual_projection = nn.Linear(visual_dim, joint_dim)
        self.audio_projection = nn.Linear(audio_dim, joint_dim)
        
        self.gate_network = nn.Sequential(
            nn.Linear(joint_dim * 2, joint_dim),
            nn.ReLU(),
            nn.Linear(joint_dim, 2), # Outputs: [Visual_Weight, Audio_Weight]
            nn.Softmax(dim=-1)       # Ensures weights add up to 1.0
        )
        
        self.ln_visual = nn.LayerNorm(joint_dim)
        self.ln_audio = nn.LayerNorm(joint_dim)

    def forward(self, visual_features: torch.Tensor, audio_features: torch.Tensor):
        v_proj = self.ln_visual(self.visual_projection(visual_features))
        a_proj = self.ln_audio(self.audio_projection(audio_features))
        
        combined_context = torch.cat([v_proj, a_proj], dim=-1)
        
        gating_weights = self.gate_network(combined_context)
        
        # THE FIX: Using [...] makes this safe for both 2D (Testing) and 3D (Pipeline) tensors
        v_weight = gating_weights[..., 0].unsqueeze(-1)
        a_weight = gating_weights[..., 1].unsqueeze(-1)
        
        fused_representation = (v_weight * v_proj) + (a_weight * a_proj)
        
        return fused_representation, (v_weight, a_weight)

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleMambaBlock(nn.Module):
    def __init__(self, d_model: int, d_state: int = 16, expand: int = 2):
        """
        Simplified State Space Model (SSM) Block
        Processes sequences in linear time O(N) by bypassing dense attention mechanisms.
        
        Args:
            d_model (int): Dimension of the input features (e.g., 256 from Phase 1)
            d_state (int): Size of the hidden state matrix
            expand (int): Expansion factor for internal processing width
        """
        super(SimpleMambaBlock, self).__init__()
        self.d_model = d_model
        self.d_state = d_state  # Save d_state explicitly to prevent splitting bugs
        d_inner = int(expand * d_model)
        
        # 1. Input Projections
        self.in_proj = nn.Linear(d_model, d_inner * 2)
        
        # 2. 1D Causal Convolution
        self.conv1d = nn.Conv1d(
            in_channels=d_inner, 
            out_channels=d_inner, 
            kernel_size=4, 
            padding=3, 
            groups=d_inner
        )
        
        # 3. State Space Parameters (1 column for delta, d_state for B, d_state for C)
        self.x_proj = nn.Linear(d_inner, 1 + 2 * d_state)
        self.dt_proj = nn.Linear(1, d_inner)
        
        # 4. Output Projection
        self.out_proj = nn.Linear(d_inner, d_model)
        
        # Normalization
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor):
        """
        Forward pass execution.
        Input shape: [Batch_Size, Sequence_Length, D_Model]
        """
        batch_size, seq_len, _ = x.shape
        residual = x
        
        # Normalize input
        x = self.norm(x)
        
        # Step 1: Project input and split into two branches (Activation & Gating)
        x_proj = self.in_proj(x)
        x_hidden, x_gate = x_proj.chunk(2, dim=-1)
        
        # Step 2: Causal Convolution
        x_hidden = x_hidden.transpose(1, 2)
        x_hidden = self.conv1d(x_hidden)[:, :, :seq_len] # Truncate padding to maintain causality
        x_hidden = x_hidden.transpose(1, 2)
        
        # Apply SiLU activation
        x_hidden = F.silu(x_hidden)
        
        # Step 3: Explicit Data-Dependent SSM Gating
        ssm_params = self.x_proj(x_hidden)
        
        # Explicitly split into 1, 16, and 16 columns. This perfectly equals 33 columns total.
        delta, B, C = torch.split(ssm_params, [1, self.d_state, self.d_state], dim=-1)
        
        # Discretize and apply
        delta = F.softplus(delta)
        gating_weight = torch.sigmoid(self.dt_proj(delta))
        
        # Modulate the hidden state with the gate
        x_fused = x_hidden * gating_weight
        
        # Step 4: Re-multiply with the original residual gate and project out
        x_out = x_fused * F.silu(x_gate)
        output = self.out_proj(x_out)
        
        return output + residual

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DualObjectiveLoss(nn.Module):
    def __init__(self, alpha: float = 0.5, temperature: float = 0.07):
        """
        Dual-Objective Loss Layer for Sequential Alignment and Soft Distillation.
        Prevents dimensional collapse by balancing fine-grained trajectory matching 
        with abstract teacher representation alignment.
        
        Args:
            alpha (float): Balancing coefficient between SCAV and Distillation losses.
            temperature (float): Scaling factor for scaling semantic similarities.
        """
        super(DualObjectiveLoss, self).__init__()
        self.alpha = alpha
        self.temperature = temperature
        self.mse_loss = nn.MSELoss()

    def compute_scav_loss(self, student_seq: torch.Tensor, target_seq: torch.Tensor) -> torch.Tensor:
        """
        Calculates sequential alignment loss using an Interpolated Euclidean sequence distance.
        Ensures temporal trajectories match without collapsing sequence variance.
        """
        # If lengths don't match exactly due to sampling rates, interpolate student to match target
        if student_seq.shape[1] != target_seq.shape[1]:
            student_seq = student_seq.transpose(1, 2) # [B, D, S_len]
            student_seq = F.interpolate(student_seq, size=target_seq.shape[1], mode='linear', align_corners=True)
            student_seq = student_seq.transpose(1, 2) # [B, S_len, D]
            
        # Normalize vectors along the feature dimension to prevent exploding gradients
        student_norm = F.normalize(student_seq, p=2, dim=-1)
        target_norm = F.normalize(target_seq, p=2, dim=-1)
        
        # Calculate element-wise mean square distance between continuous sequence trajectories
        return self.mse_loss(student_norm, target_norm)

    def compute_distillation_loss(self, student_global: torch.Tensor, teacher_global: torch.Tensor) -> torch.Tensor:
        """
        Calculates soft-constrained distillation loss via smooth cosine similarity matching.
        Avoids hard L2 constraints which trigger premature dimensional collapse.
        """
        # Normalize global representations
        student_norm = F.normalize(student_global, p=2, dim=-1)
        teacher_norm = F.normalize(teacher_global, p=2, dim=-1)
        
        # Cosine distance computation
        cosine_sim = torch.sum(student_norm * teacher_norm, dim=-1)
        distill_loss = 1.0 - cosine_sim.mean()
        
        return distill_loss

    def forward(self, student_seq: torch.Tensor, teacher_global: torch.Tensor):
        """
        Forward pass execution.
        student_seq: [Batch_Size, Seq_Len, Dimension] -> Output of Mamba Engine
        teacher_global: [Batch_Size, Dimension]       -> Output of Frozen Video Teacher
        """
        # 1. Extract global context from our student sequence (temporal pooling)
        student_global = student_seq.mean(dim=1)
        
        # 2. Reconstruct target seq from global context to verify continuous trajectory stability
        # In this self-contained block, we align the student's timeline with its own pooled target
        # to ensure local token cohesion before distilling out globally.
        target_seq_anchor = teacher_global.unsqueeze(1).expand_2d = teacher_global.unsqueeze(1).repeat(1, student_seq.shape[1], 1)
        
        loss_scav = self.compute_scav_loss(student_seq, target_seq_anchor)
        loss_distill = self.compute_distillation_loss(student_global, teacher_global)
        
        # Combined weighted loss formulation
        total_loss = (self.alpha * loss_scav) + ((1.0 - self.alpha) * loss_distill)
        
        return total_loss, loss_scav, loss_distill

In [13]:
import torch
import torch.nn as nn

class NaiveBaselineModel(nn.Module):
    def __init__(self, visual_dim: int, audio_dim: int, joint_dim: int):
        """
        The traditional baseline model. 
        It forces audio and video together without dynamic gating and 
        squashes the sequence into a single point before alignment.
        """
        super(NaiveBaselineModel, self).__init__()
        self.v_proj = nn.Linear(visual_dim, joint_dim)
        self.a_proj = nn.Linear(audio_dim, joint_dim)
        
        # Standard Multi-Layer Perceptron (MLP) for fusion instead of Mamba
        self.fusion_mlp = nn.Sequential(
            nn.Linear(joint_dim, joint_dim * 2),
            nn.ReLU(),
            nn.Linear(joint_dim * 2, joint_dim)
        )
        self.loss_fn = nn.MSELoss() # Standard rigid L2 loss (Causes collapse)

    def forward(self, visual_features: torch.Tensor, audio_seq: torch.Tensor, teacher_global: torch.Tensor):
        # 1. Standard projection
        v_proj = self.v_proj(visual_features)
        
        # 2. Temporal Pooling (This is the fatal flaw: it squashes the timeline)
        a_squashed = self.a_proj(audio_seq).mean(dim=1) 
        
        # 3. Simple addition fusion
        fused = self.fusion_mlp(v_proj + a_squashed)
        
        # 4. Standard rigid loss
        loss = self.loss_fn(fused, teacher_global)
        return loss, fused

class HybridMambaPipeline(nn.Module):
    def __init__(self, visual_dim: int, audio_dim: int, joint_dim: int):
        """
        The AIMS DTU 2026 Master Architecture.
        Integrates Adaptive Gating, Mamba Sequence Engine, and Dual-Objective Loss.
        """
        super(HybridMambaPipeline, self).__init__()
        
        # Phase 1: Adaptive Gatekeeper
        self.gatekeeper = AdaptiveGatingMechanism(visual_dim, audio_dim, joint_dim)
        
        # Phase 2: High-Speed Mamba Engine
        self.mamba_core = SimpleMambaBlock(d_model=joint_dim)
        
        # Phase 3: Dual-Objective Optimizer
        self.criterion = DualObjectiveLoss(alpha=0.6)

    def forward(self, visual_features: torch.Tensor, audio_seq: torch.Tensor, teacher_global: torch.Tensor):
        """
        Input dimensions:
        visual_features: [Batch, V_Dim] (The sparse frame)
        audio_seq: [Batch, Seq_Len, A_Dim] (The continuous audio track)
        teacher_global: [Batch, Joint_Dim] (The frozen ground truth)
        """
        batch_size, seq_len, _ = audio_seq.shape
        
        # Step 1: Expand the sparse visual frame to match the audio sequence length
        visual_expanded = visual_features.unsqueeze(1).repeat(1, seq_len, 1)
        
        # Step 2: Apply adaptive gating at every timestep
        fused_seq, gating_weights = self.gatekeeper(visual_expanded, audio_seq)
        
        # Step 3: Process the sequence chronologically via Mamba
        processed_seq = self.mamba_core(fused_seq)
        
        # Step 4: Calculate advanced loss (No squashing!)
        total_loss, loss_scav, loss_distill = self.criterion(processed_seq, teacher_global)
        
        return total_loss, processed_seq, gating_weights

In [14]:
import os
import cv2
import time
import torch
import torchaudio
import torchaudio.transforms as T
import torchvision.transforms as T_vis
import torchvision.models as models
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# --- 1. THE DATASET ENGINE ---
class CloudMultimodalDataset(Dataset):
    def __init__(self, video_paths: list, is_training=True):
        self.video_paths = video_paths
        self.is_training = is_training
        self.mel_spectrogram = T.MelSpectrogram(sample_rate=16000, n_mels=128, n_fft=1024, hop_length=512)
        
        self.augmentations = T_vis.Compose([
            T_vis.RandomHorizontalFlip(p=0.5),
            T_vis.ColorJitter(brightness=0.2, contrast=0.2)
        ]) if is_training else None

    def __len__(self): return len(self.video_paths)

    def extract_sparse_frame(self, video_path: str) -> torch.Tensor:
        try:
            cap = cv2.VideoCapture(video_path)
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) // 2)
            ret, frame = cap.read()
            cap.release()
            if not ret: return torch.zeros(3, 224, 224)
            
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, (224, 224))
            # ResNet requires standard ImageNet normalization
            frame_tensor = torch.from_numpy(frame).permute(2, 0, 1).float() / 255.0
            frame_tensor = T_vis.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])(frame_tensor)
            
            return self.augmentations(frame_tensor) if self.is_training else frame_tensor
        except: return torch.zeros(3, 224, 224)

    def extract_audio_sequence(self, video_path: str) -> torch.Tensor:
        try:
            waveform, sr = torchaudio.load(video_path, format="mp4")
            if sr != 16000: waveform = T.Resample(sr, 16000)(waveform)
            if waveform.shape[0] > 1: waveform = torch.mean(waveform, dim=0, keepdim=True)
            
            mel_spec = self.mel_spectrogram(waveform).squeeze(0).transpose(0, 1)
            mel_spec = mel_spec.unsqueeze(0).unsqueeze(0)
            mel_spec = torch.nn.functional.interpolate(mel_spec, size=(256, 128), mode='bilinear').squeeze(0).squeeze(0)
            return mel_spec
        except: return torch.zeros(256, 128)

    def __getitem__(self, idx):
        path = self.video_paths[idx]
        return self.extract_sparse_frame(path), self.extract_audio_sequence(path)


# --- 2. THE EXECUTION LOOP (TRUE DISTILLATION) ---
def run_cloud_sprint():
    print("🚀 Initializing AIMS DTU Final Cloud Sprint (ResNet Teacher Mode)...")
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    base_dir = "/kaggle/input/datasets/pevogam/ucf101" 
    all_videos = [os.path.join(r, f) for r, d, files in os.walk(base_dir) for f in files if f.endswith(('.avi', '.mp4'))]
    
    train_paths = all_videos[:5000]
    val_paths = all_videos[5000:6000]
    print(f"📚 Training on {len(train_paths)} videos. 🧪 Validating on {len(val_paths)} videos.")
    
    train_loader = DataLoader(CloudMultimodalDataset(train_paths, is_training=True), batch_size=16, shuffle=True, num_workers=2)
    val_loader = DataLoader(CloudMultimodalDataset(val_paths, is_training=False), batch_size=16, shuffle=False, num_workers=2)
    
    # 1. Initialize the Student
    model = HybridMambaPipeline(visual_dim=512, audio_dim=128, joint_dim=256).to(DEVICE)
    visual_projector = torch.nn.Linear(3 * 224 * 224, 512).to(DEVICE)
    
    # 2. Initialize the Frozen Teacher (ResNet-18)
    print("⏳ Downloading Frozen ResNet Teacher...")
    teacher_model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1).to(DEVICE)
    teacher_model.fc = torch.nn.Identity() # Strip the final classification layer to get 512-D features
    teacher_model.eval() # Freeze the teacher
    print("✅ Teacher Locked.")
    
    optimizer = optim.AdamW(list(model.parameters()) + list(visual_projector.parameters()), lr=3e-4, weight_decay=1e-3)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
    
    for epoch in range(10):
        model.train()
        start_time = time.time()
        print(f"\n--- EPOCH {epoch+1}/10 STARTED ---")
        
        for batch_idx, (v_frames, a_seqs) in enumerate(train_loader):
            v_frames, a_seqs = v_frames.to(DEVICE), a_seqs.to(DEVICE)
            optimizer.zero_grad()
            
            # The Student's internal view
            v_feats = visual_projector(v_frames.flatten(start_dim=1))
            
            # THE FIX: The true, unmoving target from the Frozen Teacher
            with torch.no_grad():
                true_teacher_target = teacher_model(v_frames)[:, :256].detach()
            
            loss, out_seq, _ = model(v_feats, a_seqs, true_teacher_target)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            if batch_idx % 100 == 0:
                print(f"  > Batch {batch_idx} processed. Loss: {loss.item():.4f}")
                
        scheduler.step()
        
        # --- ZERO-SHOT VALIDATION ---
        model.eval()
        val_cos_sim = 0.0
        with torch.no_grad():
            for v_frames, a_seqs in val_loader:
                v_frames, a_seqs = v_frames.to(DEVICE), a_seqs.to(DEVICE)
                v_feats = visual_projector(v_frames.flatten(start_dim=1))
                
                # Compare against the Teacher
                true_teacher_target = teacher_model(v_frames)[:, :256].detach()
                
                _, out_seq, _ = model(v_feats, a_seqs, true_teacher_target)
                
                out_global = F.normalize(out_seq.mean(dim=1), p=2, dim=-1)
                target_norm = F.normalize(true_teacher_target, p=2, dim=-1)
                val_cos_sim += torch.sum(out_global * target_norm, dim=-1).mean().item()
                
        avg_val_cos = val_cos_sim / len(val_loader)
        epoch_time = (time.time() - start_time) / 60
        print(f"✅ EPOCH {epoch+1} COMPLETE | Time: {epoch_time:.1f}m | Teacher Distillation Cosine Sim: {avg_val_cos:.4f}")

if __name__ == "__main__":
    run_cloud_sprint()

🚀 Initializing AIMS DTU Final Cloud Sprint (ResNet Teacher Mode)...
📚 Training on 5000 videos. 🧪 Validating on 1000 videos.
⏳ Downloading Frozen ResNet Teacher...
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 164MB/s] 


✅ Teacher Locked.

--- EPOCH 1/10 STARTED ---


/usr/local/lib/python3.12/dist-packages/torchaudio/__init__.py:86: UserWarning: The 'format' parameter is not supported by TorchCodec AudioDecoder.
  return load_with_torchcodec(
/usr/local/lib/python3.12/dist-packages/torchaudio/__init__.py:86: UserWarning: The 'format' parameter is not supported by TorchCodec AudioDecoder.
  return load_with_torchcodec(


  > Batch 0 processed. Loss: 0.3985
  > Batch 100 processed. Loss: 0.0713
  > Batch 200 processed. Loss: 0.0756
  > Batch 300 processed. Loss: 0.0682


...

✅ EPOCH 9 COMPLETE | Time: 9.5m | Teacher Distillation Cosine Sim: 0.4342

--- EPOCH 10/10 STARTED ---


/usr/local/lib/python3.12/dist-packages/torchaudio/__init__.py:86: UserWarning: The 'format' parameter is not supported by TorchCodec AudioDecoder.
  return load_with_torchcodec(
/usr/local/lib/python3.12/dist-packages/torchaudio/__init__.py:86: UserWarning: The 'format' parameter is not supported by TorchCodec AudioDecoder.
  return load_with_torchcodec(


  > Batch 0 processed. Loss: 0.0554
  > Batch 100 processed. Loss: 0.0525
  > Batch 200 processed. Loss: 0.0531
  > Batch 300 processed. Loss: 0.0417


/usr/local/lib/python3.12/dist-packages/torchaudio/__init__.py:86: UserWarning: The 'format' parameter is not supported by TorchCodec AudioDecoder.
  return load_with_torchcodec(
/usr/local/lib/python3.12/dist-packages/torchaudio/__init__.py:86: UserWarning: The 'format' parameter is not supported by TorchCodec AudioDecoder.
  return load_with_torchcodec(


✅ EPOCH 10 COMPLETE | Time: 9.5m | Teacher Distillation Cosine Sim: 0.4320
